# Importing micro and macro data.
- Functions *load_lamp* and *load_macro*.
- At this point, there is quite a lot of hardcoded variables here. Therefore, it wouldn't work with a different model. 
- In the next project we can work more with the input data (both coefficients from stats model and micro/macrolevel data) so that load functions are simpler and coherent. 

## 1. Generate a function to load micro data. 
- Variables to be selected are defined by the models that were considered in the statistical model. 

In [2]:
function load_lamp(path_stats, path_micro) 
	init_var = 3
	end_var = 34

	stats = CSV.read(path_stats, DataFrame)
	vars_interest =  stats[init_var:end_var, 1]

	data = CSV.read(path_micro, DataFrame)
	data = select(data, "id_pers", "year", vars_interest, "weight")
	lamp = Dict{String, Vector}()

	for row in eachrow(data)
		if haskey(lamp, row[1])
			push!(lamp[row[1]], row[2:end])
	    else
			lamp[row[1]] = [ row[2:end] ]
		end
	end
	start_micro = minimum(map(x -> x[1][1], values(lamp)))
	end_micro = maximum(map(x -> x[1][1], values(lamp)))
	lamp, start_micro, end_micro
end

load_lamp (generic function with 1 method)

1.1. Data transformations and functions to handle data better.

In [3]:
# person_years is a Vector{DataFrameRow} from the LAMP data
weight(person_years) = round(Int, person_years[1][end])
birth_year(person_years) = person_years[1][1]

birth_year (generic function with 1 method)

This makes life easier down the line, plus the program more efficient. Also,
this way we can cut out year and weight, so values is really only values:

In [4]:
"convert list of person years from LAMP data (DataFrameRow) to Vector{Float}"
lamp_to_values(person_years) = map(x->Vector{Float64}(x)[2:end-1], person_years)

lamp_to_values

## 2. Function to load macro data.
- Read variables from statistical model output and select the ones that corresponde to macro data.

In [5]:
function load_macro(path_stats, path_macro) 

	init_var = 35
	end_var = init_var + 5

	stats = CSV.read(path_stats, DataFrame)
	vars_interest =  stats[init_var:end_var, 1]

	data_macro = CSV.read(path_macro, DataFrame)
	macro_values = Array(select(data_macro, "year", vars_interest))
	start_macro = minimum(macro_values[:,1])
	end_macro = maximum(macro_values[:,1])
macro_values, start_macro, end_macro
end



load_macro (generic function with 1 method)

## 3. Load the statistical model
- Factors (stats model coefficients) are selected depending on the destination of interest.
- We only need the position of the first factor. 

In [6]:
function load_stats(path, dest)
	if dest == "Intraregional"
		init_micro_var = 43
	elseif dest == "USA"
		init_micro_var = 83
	elseif dest == "Spain"
		init_micro_var = 123
	end

   end_micro_var = init_micro_var + 31
   init_macro_var = end_micro_var + 1
   end_macro_var = init_macro_var + 5
   stats = CSV.read(path, DataFrame)
   micro_factors = parse.(Float64, stats[init_micro_var:end_micro_var, 2])
   macro_factors = parse.(Float64, stats[init_macro_var:end_macro_var, 2])
   offset = parse.(Float64, stats[end,2])
   micro_factors, macro_factors, offset
end

load_stats (generic function with 1 method)